# SAADE–AUA v3.1 — попередній аудит підключення (без навчання)

Це **читання** `audio_real_fake/AUA` і звіт у `audio_real_fake/results/SAADE_AUA_2026_v3_1_metadata_fixed`. Каталог `Melnyk_dus` не змінюється. `05_splits` і первинні архіви в `00_sources_and_licenses` не включаються як нові незалежні приклади.

## Незалежний аудит AUA, лише читання

Раніше виконані аудити з `/home/jovyan/work/Melnyk_dus/results/AUA_audit/deep_audit` використовуються лише як довідкові матеріали. Жодні файли не створюються й не змінюються в цій папці. Відсутність файлів не блокує SAADE.

In [ ]:
from pathlib import Path
AUDIT_DIR=Path('/home/jovyan/work/Melnyk_dus/results/AUA_audit/deep_audit')
for filename in ['AUA_DEEP_AUDIT_SUMMARY.txt','FINAL_KPI_ACCEPTANCE.csv','AUA_FULL_AUDIT.xlsx']:
    file=AUDIT_DIR/filename
    print(filename,'—',('є, тільки читання' if file.is_file() else 'недоступний на цьому сервері'))
# Ця комірка тільки перевіряє існування довідкових звітів. Вона нічого не записує.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display
PROJECT_ROOT=Path('/home/jovyan/work/audio_real_fake')
DATA_ROOT=PROJECT_ROOT/'AUA'
RESULTS_ROOT=PROJECT_ROOT/'results'/'SAADE_AUA_2026_v3_1_metadata_fixed'
assert DATA_ROOT.is_dir(),DATA_ROOT
assert RESULTS_ROOT.resolve().is_relative_to((PROJECT_ROOT/'results').resolve())
assert not any('melnyk' in part.lower() for part in RESULTS_ROOT.resolve().parts)
sys.path.insert(0,str(Path.cwd()))
import saade_v3_core as S
import saade_v3_data as D
cfg=S.Config(root=str(DATA_ROOT),out=str(RESULTS_ROOT),scenario_profile='v2_5level')
print('AUA:',cfg.root,'\nOUTPUT:',cfg.out)


In [ ]:
catalog=D.build_verified_catalog(cfg)
print('Знайдено:',len(catalog),'придатних:',int(catalog.usable.sum()))
display(pd.crosstab(catalog.source,catalog.label,dropna=False))
display(pd.read_csv(RESULTS_ROOT/'01_sources_summary.csv'))
display(catalog.groupby('source').agg(files=('path','size'),groups=('group_id','nunique'),usable=('usable','sum')))


In [ ]:
d=catalog[catalog.source.eq('DRSSU')]
print('DRSSU всього:',len(d),'метадані зв’язані:',int(d.primary_source_meta_found.sum()),
      'fake:',int(d.label.eq(1).sum()),'usable:',int(d.usable.sum()))
print('Дублікати первинного SHA:',int(d.warning.fillna('').str.contains('DRSSU_DUPLICATE_SOURCE_SHA').sum()))
display(d[['relative_path','source_record_id','label','label_source','split_role','warning']].head(10))
assert d.empty or d.primary_source_meta_found.any(), 'Помилка зв’язування DRSSU: source_record_id ↔ selection_rank'


**Інтерпретація:** якщо DRSSU успішно пов’язано, але має `split_role=train_only`, це правильно за відсутності підтверджених голосових ID. Експертну оцінку й незалежний hidden test цей аудит не виконує. Для експерименту запускайте основний ноутбук.